## Data Cleaning Script

### Packages

In [16]:
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
from io import StringIO
from shapely import wkt, distance, LineString

### Reading the Data

In [ ]:
url = "https://raw.githubusercontent.com/iansargent/nyc-subway-ridership-ml/refs/heads/main/Data/Raw/origin_destination_flows.csv"
od_flows_raw = pd.read_csv(StringIO(requests.get(url, verify=False).text))
od_flows = od_flows_raw.copy()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


178018

### Cleaning

#### Column Names

In [3]:
# Standardize column names
od_flows.columns = od_flows_raw.columns.str.strip().str.lower().str.replace(" ", "_")

#### Data Types

In [4]:
# Turn the ridership column to numeric (remove commas)
od_flows['sum_estimated_average_ridership'] = (
    pd.to_numeric(od_flows_raw['sum_estimated_average_ridership'].str.replace(',', ''))
)

#### Turn the Point Data into a Geometry Type

In [5]:
od_flows["origin_point"] = wkt.loads(od_flows["origin_point"])
od_flows["destination_point"] = wkt.loads(od_flows["destination_point"])

#### Turn into a GeoDataFrame for Mapping

In [6]:
od_flows_geo = gpd.GeoDataFrame(od_flows)

#### Calculate Distance Between Two Stations (Meters + Kilometers)

In [7]:
# Map projection for distance calculations
crs = "EPSG:32618" 

# Change projection of origin point
od_flows_geo["origin_point"] = gpd.GeoSeries(
    od_flows_geo["origin_point"], crs="EPSG:4326"
).to_crs(crs)

# Change projection of destination point
od_flows_geo["destination_point"] = gpd.GeoSeries(
    od_flows_geo["destination_point"], crs="EPSG:4326"
).to_crs(crs)

# Calculatio meter distance between the two points
od_flows_geo["distance_meters"] = (
    gpd.GeoSeries(od_flows_geo["origin_point"])
    .distance(gpd.GeoSeries(od_flows_geo["destination_point"]))
)

# Create a kilometers column
od_flows_geo["distance_km"] = od_flows_geo["distance_meters"] / 1000

#### Log of Ridership (skewed)

In [8]:
od_flows_geo["log_ridership"] = np.log1p(od_flows_geo["sum_estimated_average_ridership"])

#### Riders per Kilometer Variable

In [14]:
od_flows_geo["riders_per_km"] = (
    od_flows_geo["sum_estimated_average_ridership"] /
    od_flows_geo["distance_km"]
)

od_flows_geo.columns

Index(['origin_station_complex_name', 'destination_station_complex_name',
       'origin_station_complex_id', 'destination_station_complex_id',
       'origin_point', 'destination_point', 'sum_estimated_average_ridership',
       'distance_meters', 'distance_km', 'log_ridership', 'riders_per_km',
       'same_station'],
      dtype='object')

In [18]:
od_flows_geo["geometry"] = od_flows_geo.apply(
    lambda row: LineString([row["origin_point"], row["destination_point"]]), axis=1
)

od_flows_geo["origin_point_wkt"] = od_flows_geo["origin_point"].apply(lambda g: g.wkt)
od_flows_geo["destination_point_wkt"] = od_flows_geo["destination_point"].apply(lambda g: g.wkt)


od_flows_geo.drop(columns=["origin_point", "destination_point"]) \
    .set_geometry("geometry") \
    .to_file("origin_destination_flows_CLEAN.geojson", driver="GeoJSON")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pyogrio/geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
